# Supplementary Figure — across-condition versus within-condition full MS-AA landscapes

This figure reproduces the full-reconstruction comparison between shared
across-condition and separate within-condition fits at matched normalized
representation size.

Rows distinguish fit scope (across vs within); columns distinguish spatial
versus temporal MS-AA. Condition colors follow the manuscript:
purple = intact, green = word-scrambled, black = rest.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CONDITIONS = ["intact", "word", "rest"]
COLORS = {
    "intact": "#7B3294",
    "word": "#2CA25F",
    "rest": "#222222",
}

FILES = {
    ("spatial", "across"): Path(
        "msaa_condrank_decoding_outputs_spatial_across/full_summary.csv"
    ),
    ("spatial", "within"): Path(
        "msaa_condrank_decoding_outputs_spatial_within/full_summary.csv"
    ),
    ("temporal", "across"): Path(
        "msaa_condrank_decoding_outputs_temporal_across/full_summary.csv"
    ),
    ("temporal", "within"): Path(
        "msaa_condrank_decoding_outputs_temporal_within/full_summary.csv"
    ),
}

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_full_across_within_ratio_landscapes.png"
FIG_PDF = OUTPUT_DIR / "Supplement_full_across_within_ratio_landscapes.pdf"

tables = {}

for key, path in FILES.items():
    if not path.exists():
        print("Missing:", path)
        continue

    df = pd.read_csv(path)

    analysis_type, fit_scope = key
    df["analysis_type"] = analysis_type
    df["fit_scope"] = fit_scope

    if "component_ratio" not in df.columns:
        denom = 700 if analysis_type == "spatial" else 300
        df["component_ratio"] = df["K"].astype(float) / denom

    tables[key] = df

if len(tables) == 0:
    raise FileNotFoundError(
        "No full_summary.csv tables were found. Run 002_decode_msaa for "
        "the desired spatial/temporal × across/within combinations."
    )

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.2, 8.2),
    sharex=True,
)

layout = [
    ("spatial", "across"),
    ("temporal", "across"),
    ("spatial", "within"),
    ("temporal", "within"),
]

titles = {
    ("spatial", "across"): "Spatial MS-AA — across-condition basis",
    ("temporal", "across"): "Temporal MS-AA — across-condition basis",
    ("spatial", "within"): "Spatial MS-AA — within-condition fits",
    ("temporal", "within"): "Temporal MS-AA — within-condition fits",
}

for ax, key, letter in zip(
    axes.ravel(),
    layout,
    list("ABCD"),
):
    df = tables.get(key)

    if df is None:
        ax.text(
            0.5, 0.5,
            "Output not available",
            ha="center", va="center",
            transform=ax.transAxes,
        )
        ax.set_axis_off()
        continue

    for condition in CONDITIONS:
        cur = (
            df[
                df["condition"].astype(str) == condition
            ]
            .sort_values("component_ratio")
        )

        if len(cur) == 0:
            continue

        ax.errorbar(
            cur["component_ratio"],
            cur["mean"],
            yerr=cur["err"] if "err" in cur.columns else None,
            color=COLORS[condition],
            marker="o",
            markersize=3.2,
            linewidth=1.8,
            capsize=2,
            label=condition.capitalize(),
        )

    ax.set_title(titles[key], fontweight="bold", fontsize=12)
    ax.set_xlabel("Normalized representational ratio")
    ax.set_ylabel("Timepoint-identification accuracy")
    ax.tick_params(labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )

handles = [
    plt.Line2D([0], [0], color=COLORS[c], lw=2, label=c.capitalize())
    for c in CONDITIONS
]

fig.legend(
    handles=handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.suptitle(
    "Full MS-AA decoding across fit scopes and orientations",
    fontsize=16,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    bottom=0.10,
    top=0.93,
    hspace=0.30,
    wspace=0.24,
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)